In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

img = Image.open("waveform.jpeg").convert("L")
gray = np.array(img)

plt.figure(figsize=(100, 50))
plt.imshow(gray, cmap="gray")
plt.axis("off")
plt.tight_layout()
plt.show()


## step 1 — okay what does this thing even look like

loading the image and converting it to grayscale. nothing fancy. each pixel is now just a number from 0 (black) to 255 (white). yes the figure is enormous. the image is big. deal with it.

# 🔔 Albert Gaussian — Episode ??: My Friend Made Music and There Were Gaussians In It

okay so. my friend sent me his new track as a waveform image. normal person reaction: "sounds cool bro". my reaction: *opens jupyter notebook*.

this is albert gaussian. i am alper sarıtaş. in german that's approximately albert gelbstein which is approximately albert gaussian and i love gaussians so much that i made it my whole personality. this channel/project/bit is about finding gaussians hiding in everyday real life and going "SEE!! IT'S EVERYWHERE!!"

today's victim: a jpeg of a music waveform. let's ruin it with math.

**what we're doing:**
1. look at the waveform image
2. figure out a threshold to make it black & white
3. detect all the little bar shapes in it
4. collect their heights
5. discover (spoiler) there are gaussians in there
6. lose our minds about it


In [ ]:
plt.figure(figsize=(12, 4))
plt.bar(range(256), np.bincount(gray.ravel(), minlength=256), width=1, color='steelblue')
plt.xlabel('Pixel value (0–255)')
plt.ylabel('Count')
plt.xticks(range(0, 256, 10), rotation=90, fontsize=7)
plt.tight_layout()
plt.show()


## step 2 — let's spy on every single pixel

plot every pixel value from 0 to 255. we're looking for the gap between "dark background" and "bright bar". that gap is where we cut. find the valley, that's your number.

In [ ]:
import ipywidgets as widgets

threshold_slider = widgets.IntSlider(value=180, min=0, max=255, step=1,
                                     description='Threshold:',
                                     layout=widgets.Layout(width='400px'))
display(threshold_slider)


## step 3 — the great pixel sorting ceremony

pixel above threshold? WHITE. pixel below threshold? BLACK. that's it. that's the whole algorithm. drag the slider until the bars look clean and not merged into a blob. re-run after moving it, the slider doesn't do magic on its own.

In [ ]:
binary = (gray >= threshold_slider.value).astype(np.uint8) * 255

plt.figure(figsize=(120, 40))
plt.imshow(binary, cmap='gray', interpolation='nearest')
plt.title(f'Threshold = {threshold_slider.value}')
plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
from scipy.ndimage import label
import matplotlib.patches as patches

# Label every contiguous white region (4-connectivity)
struct = np.array([[0,1,0],[1,1,1],[0,1,0]])  # 4-connected neighbours
labeled, n_chunks = label(binary == 255, structure=struct)

# Bounding box per chunk
boxes = []
for i in range(1, n_chunks + 1):
    rows, cols = np.where(labeled == i)
    boxes.append({
        'id':     i,
        'top':    int(rows.min()),
        'bottom': int(rows.max()),
        'left':   int(cols.min()),
        'right':  int(cols.max()),
        'width':  int(cols.max() - cols.min() + 1),
        'height': int(rows.max() - rows.min() + 1),
        'pixels': len(rows),
    })

fig, ax = plt.subplots(figsize=(140, 50))
ax.imshow(binary, cmap='gray', interpolation='nearest')
for b in boxes:
    ax.add_patch(patches.Rectangle(
        (b['left'] - 0.5, b['top'] - 0.5), b['width'], b['height'],
        linewidth=5, edgecolor='red', facecolor='none'
    ))
ax.set_title(f'{n_chunks} chunks detected')
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Chunks: {n_chunks}")


## step 4 — finding the bars (the music kind, not the drinking kind)

we walk through every white pixel and ask "are any of your neighbours also white?" — if yes, same group. this is called 4-connectivity which is a fancy way of saying we don't count diagonals because we're not monsters. each blob gets a red bounding box. if everything is one giant box the threshold is too low, go back and fix it.

In [ ]:
bins_slider = widgets.IntSlider(value=50, min=1, max=300, step=1,
                                description='Bins:',
                                layout=widgets.Layout(width='400px'))
display(bins_slider)


## step 5 — the height histogram (this is where it gets interesting)

we grab the pixel height of every box and dump it into a histogram. you'll notice it's not one clean peak — it's two. one for the tiny jitter/noise blobs and one for the actual bars. two populations. two modes. a bimodal distribution staring right at us. and you know what lives inside bimodal distributions? that's right. gaussians. GAUSSIANS.

In [ ]:
heights = [b['height'] for b in boxes]

plt.figure(figsize=(12, 4))
plt.hist(heights, bins=bins_slider.value, color='steelblue', edgecolor='none')
plt.xlabel('Box height (px)')
plt.ylabel('Count')
plt.title(f'Height distribution — {len(heights)} boxes, {bins_slider.value} bins')
plt.tight_layout()
plt.show()


In [ ]:
n_components_slider = widgets.IntSlider(value=3, min=1, max=10, step=1,
                                        description='Components:',
                                        layout=widgets.Layout(width='400px'))
display(n_components_slider)


## step 6 — THERE THEY ARE. THE GAUSSIANS.

we fit a Gaussian Mixture Model to the height distribution. it uses an algorithm called EM (Expectation-Maximization) which sounds intimidating but basically just jiggles the gaussians around until they fit. BIC (Bayesian Information Criterion) tells us how many gaussians to use without overfitting — lower BIC = better fit. the left plot shows BIC vs component count, the right shows the actual fitted curves.

my friend just wanted me to listen to his song. instead i found gaussians in it. he's going to be so confused.

In [ ]:
from sklearn.mixture import GaussianMixture
from scipy.stats import norm
import matplotlib.cm as cm

X = np.array(heights).reshape(-1, 1)

# BIC curve (reference)
bics, models = [], []
for k in range(1, 11):
    g = GaussianMixture(n_components=k, random_state=42, max_iter=300)
    g.fit(X)
    bics.append(g.bic(X))
    models.append(g)

bic_best_n = np.argmin(bics) + 1

# Use slider value for the actual fit
n_chosen = n_components_slider.value
best = models[n_chosen - 1]
means   = best.means_.flatten()
stds    = np.sqrt(best.covariances_.flatten())
weights = best.weights_.flatten()

x = np.linspace(min(heights), max(heights), 1000)
colors = cm.tab10(np.linspace(0, 1, n_chosen))

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

# BIC curve
axes[0].plot(range(1, 11), bics, marker='o', color='steelblue')
axes[0].axvline(bic_best_n, color='gray', linestyle='--', linewidth=1, label=f'BIC best = {bic_best_n}')
axes[0].axvline(n_chosen,   color='red',  linestyle='--', linewidth=1, label=f'Selected = {n_chosen}')
axes[0].set_xlabel('n components')
axes[0].set_ylabel('BIC')
axes[0].set_title('BIC curve')
axes[0].legend(fontsize=8)

# GMM fit
axes[1].hist(heights, bins=bins_slider.value, density=True,
             color='steelblue', edgecolor='none', alpha=0.5)
for i, (m, s, w) in enumerate(zip(means, stds, weights)):
    axes[1].plot(x, w * norm.pdf(x, m, s), color=colors[i], linewidth=1.5,
                 label=f'G{i+1} μ={m:.0f} σ={s:.0f} w={w:.2f}')
total = sum(w * norm.pdf(x, m, s) for m, s, w in zip(means, stds, weights))
axes[1].plot(x, total, color='green', linewidth=1.5, linestyle='--', label='Mixture')
axes[1].set_xlabel('Box height (px)')
axes[1].set_ylabel('Density')
axes[1].set_title(f'GMM fit  (n={n_chosen})')
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

for i, (m, s, w) in enumerate(zip(means, stds, weights)):
    print(f'G{i+1}  μ={m:.2f}  σ={s:.2f}  weight={w:.3f}')


In [ ]:
n_slider = widgets.IntSlider(value=2, min=1, max=5, step=1,
                             description='N samples:',
                             layout=widgets.Layout(width='500px'))
display(n_slider)


## step 7 — and now for something completely unnecessary (Central Limit Theorem)

we already found our gaussians. we're done. but let's also prove the Central Limit Theorem for fun because why not. take N random heights, average them. do that 1000 times. plot the results. at N=1 you see the original chaotic bimodal mess. crank N up and watch it magically collapse into a perfect gaussian. every time. no matter what the original distribution looked like. it always becomes a gaussian. this is why albert gaussian exists. it's always a gaussian. it's ALWAYS a gaussian.

In [ ]:
from scipy.stats import norm

N      = n_slider.value
TRIALS = 1000
rng    = np.random.default_rng(42)

# Draw TRIALS means, each mean is the average of N samples from the height distribution
sample_means = rng.choice(heights, size=(TRIALS, N), replace=True).mean(axis=1)

# Fit a Gaussian to the sample means
mu, sigma = norm.fit(sample_means)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(sample_means, bins=80, density=True, color='steelblue', edgecolor='none', alpha=0.7)

x = np.linspace(sample_means.min(), sample_means.max(), 300)
ax.plot(x, norm.pdf(x, mu, sigma), color='red', linewidth=1.5, label=f'Gaussian fit  μ={mu:.1f}  σ={sigma:.1f}')

ax.set_xlabel('Mean box height (px)')
ax.set_ylabel('Density')
ax.set_title(f'CLT simulation — N={N} samples per trial, {TRIALS} trials\n'
             f'Original distribution is bimodal; means converge to Gaussian as N grows')
ax.legend()
plt.tight_layout()
plt.show()
